In [0]:
%pip install azure-identity azure-mgmt-databricks azure-mgmt-subscription --quiet
dbutils.library.restartPython()

In [0]:
from azure.identity import ClientSecretCredential
from azure.mgmt.subscription import SubscriptionClient
from azure.mgmt.databricks import AzureDatabricksManagementClient

# Read credentials from Databricks Secrets (scope: azure-sp)
tenant_id = dbutils.secrets.get(scope="azure-sp", key="tenant-id")
client_id = dbutils.secrets.get(scope="azure-sp", key="client-id")
client_secret = dbutils.secrets.get(scope="azure-sp", key="client-secret")

credential = ClientSecretCredential(tenant_id, client_id, client_secret)
print("✅ Authenticated via Service Principal (credentials from Databricks Secrets)")

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
w.secrets.put_secret(scope="azure-sp", key="client-secret", string_value="xxxx")

In [0]:
# List all Azure subscriptions
sub_client = SubscriptionClient(credential)
subscriptions = list(sub_client.subscriptions.list())
print(f"Found {len(subscriptions)} subscriptions:")
for sub in subscriptions:
    print(f"  {sub.subscription_id}: {sub.display_name}")

In [0]:
# Extract Databricks workspaces from ALL subscriptions
# Store as list of dicts to capture all attributes
workspace_list = []

for sub in subscriptions:
    sub_id = sub.subscription_id
    sub_name = sub.display_name
    try:
        dbr_client = AzureDatabricksManagementClient(credential, sub_id)
        ws_list = list(dbr_client.workspaces.list_by_subscription())
        for ws in ws_list:
            workspace_list.append({
                "workspace_id": str(ws.workspace_id),
                "workspace_name": ws.name,
                "subscription_name": sub_name,
                "resource_group": ws.id.split("/resourceGroups/")[1].split("/")[0] if ws.id and "/resourceGroups/" in ws.id else None,
                "location": ws.location,
            })
        print(f"  {sub_name} ({sub_id}): {len(ws_list)} workspaces")
    except Exception as e:
        print(f"  {sub_name} ({sub_id}): error — {e}")

print(f"\n✅ Total workspaces found: {len(workspace_list)}")
for ws in sorted(workspace_list, key=lambda x: x['workspace_name']):
    print(f"  {ws['workspace_id']}: {ws['workspace_name']} [{ws['subscription_name']}] [{ws['resource_group']}] [{ws['location']}]")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("workspaceid", StringType(), True),
    StructField("workspacename", StringType(), True),
    StructField("subscription_name", StringType(), True),
    StructField("resource_group", StringType(), True),
    StructField("location", StringType(), True),
])

# Build rows from workspace_list dicts
rows = [(ws["workspace_id"], ws["workspace_name"], ws["subscription_name"], ws["resource_group"], ws["location"]) for ws in workspace_list]

df = spark.createDataFrame(rows, schema)
display(df)

In [0]:
# Write to bronze table (with expanded schema)
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.default.databricks_workspaces")

print(f"Wrote {df.count()} workspaces to bronze.default.databricks_workspaces")

# Add subscription_name column to mapping table if it doesn't exist
try:
    spark.sql("ALTER TABLE salama_insurance.salama_silver.workspace_mapping ADD COLUMNS (subscription_name STRING)")
    print("✅ Added subscription_name column to workspace_mapping")
except Exception as e:
    if "already exists" in str(e).lower():
        print("ℹ️ subscription_name column already exists")
    else:
        print(f"⚠️ Column add note: {e}")

# Sync into the FinOps mapping table
try:
    spark.sql("""
    MERGE INTO salama_insurance.salama_silver.workspace_mapping AS target
    USING bronze.default.databricks_workspaces AS source
    ON target.workspace_id = source.workspaceid
    WHEN MATCHED THEN
        UPDATE SET
            workspace_name = source.workspacename,
            subscription_name = source.subscription_name,
            resource_group = source.resource_group,
            location = source.location
    WHEN NOT MATCHED THEN
        INSERT (workspace_id, workspace_name, subscription_name, resource_group, location)
        VALUES (source.workspaceid, source.workspacename, source.subscription_name, source.resource_group, source.location)
    """)
    print("✅ Synced workspace details to salama_insurance.salama_silver.workspace_mapping")
except Exception as e:
    print(f"⚠️ Mapping table sync skipped: {e}")

In [0]:
%sql

select * from bronze.default.databricks_workspaces

In [0]:
# Verify the updated workspace mapping
print("Updated workspace mapping:")
spark.sql("""
SELECT workspace_id, workspace_name, subscription_name, resource_group, location
FROM salama_insurance.salama_silver.workspace_mapping 
ORDER BY subscription_name, workspace_name
""").show(20, truncate=False)